## Colab Test

this file is for test reactant on the simple example of computing pie via a numerical method.

In [1]:
5-1

4

In [ ]:
cd('BASEforHANK_Cloud_CPU_GPU_TPU')

In [2]:
println(pwd()) 

/Users/max/colab/BASEforHANK_Cloud_CPU_GPU_TPU


In [ ]:
println("Julia Version Running: ", VERSION)

Julia Version Running: 1.11.5


In [27]:
import Pkg

In [28]:
Pkg.add("Metal") 

   Resolving package versions...
   Installed LLVMDowngrader_jll ─ v0.6.0+1
   Installed UnsafeAtomics ────── v0.3.1
   Installed Random123 ────────── v1.7.1
   Installed KernelAbstractions ─ v0.9.41
   Installed RandomNumbers ────── v1.6.0
   Installed Atomix ───────────── v1.1.3
   Installed BFloat16s ────────── v0.6.1
   Installed GPUArrays ────────── v11.4.1
   Installed GPUToolbox ───────── v1.1.1
   Installed ObjectiveC ───────── v3.4.2
   Installed Metal ────────────── v1.9.3
      Compat entries added for Metal
    Updating `~/colab/BASEforHANK_Cloud_CPU_GPU_TPU/Project.toml`
  [dde4c033] + Metal v1.9.3
    Updating `~/colab/BASEforHANK_Cloud_CPU_GPU_TPU/Manifest.toml`
  [a9b6321e] + Atomix v1.1.3
  [ab4f0b2a] + BFloat16s v0.6.1
  [0c68f7d7] + GPUArrays v11.4.1
  [096a3bc2] + GPUToolbox v1.1.1
  [63c18a36] + KernelAbstractions v0.9.41
  [dde4c033] + Metal v1.9.3
  [e86c9b32] + ObjectiveC v3.4.2
  [74087812] + Random123 v1.7.1
  [e6cf234a] + RandomNumbers v1.6.0
  [013be700] + U

In [ ]:
Pkg.add("BenchmarkTools")
Pkg.add("Reactant")
Pkg.add("Random")
Pkg.add("PyCall") 
Pkg.add("Metal") 

    Updating registry at `~/.julia/registries/General`
┌ Info: The General registry is installed via git. Consider reinstalling it via
│ the newer faster direct from tarball format by running:
│   pkg> registry rm General; registry add General
│ 
└ @ Pkg.Registry /Applications/Julia-1.11.app/Contents/Resources/julia/share/julia/stdlib/v1.11/Pkg/src/Registry/Registry.jl:478
    Updating git-repo `https://github.com/JuliaRegistries/General.git`
   Resolving package versions...
   Installed Cairo_jll ──── v1.18.6+0
   Installed QuadGK ─────── v2.11.3
   Installed AbstractMCMC ─ v5.15.1
      Compat entries added for BenchmarkTools
    Updating `~/colab/BASEforHANK_Cloud_CPU_GPU_TPU/Project.toml`
  [37e2e46d] ↓ LinearAlgebra v1.12.0 ⇒ v1.11.0
  [2f01184e] ↓ SparseArrays v1.12.0 ⇒ v1.11.0
    Updating `~/colab/BASEforHANK_Cloud_CPU_GPU_TPU/Manifest.toml`
  [80f14c24] ↑ AbstractMCMC v5.14.0 ⇒ v5.15.1
  [9718e550] - Baselet v0.1.1
  [244e2a9f] - DefineSingletons v0.1.2
  [ac6e5ff7] - JuliaSyn

In [2]:
import Pkg, Reactant, BenchmarkTools, Random

In [ ]:
using BenchmarkTools, PyCall, CSV, DataFrames, Metal

In [30]:
Threads.nthreads()
Metal.versioninfo()

macOS 14.5.0, Darwin 23.5.0

Toolchain:
- Julia: 1.11.9
- LLVM: 16.0.6

Julia packages:
- Metal.jl: 1.9.3
- GPUArrays: 11.4.1
- GPUCompiler: 1.9.1
- KernelAbstractions: 0.9.41
- ObjectiveC: 3.4.2
- LLVM: 9.4.6
- LLVMDowngrader_jll: 0.6.0+1

1 device:
- Apple M2 10 GPU cores (64.000 KiB allocated)


In [31]:
N = 10^7
x = MtlArray(rand(Float32, N))
y = MtlArray(rand(Float32, N))

inside = @. ifelse(x^2 + y^2 <= 1f0, 1f0, 0f0)
pi_est = 4f0 * sum(inside) / Float32(N)

pi_est

3.141448f0

In [23]:
# NamedTuple results that holds all relevant information about the current Julia version and the name "Anna"
# and the run times of the benchmarks
results = (KERNEL = Sys.KERNEL,
            MACHINE = Sys.MACHINE,
            CPU_NAME = Sys.CPU_NAME,
            JULIA_VERSION = VERSION,
            nthreads = Threads.nthreads(),
            TOTAL_MEMORY = Sys.total_memory() |> Base.format_bytes,
            FREE_MEMORY = Sys.free_memory() |> Base.format_bytes )

(KERNEL = :Darwin, MACHINE = "arm64-apple-darwin24.0.0", CPU_NAME = "apple-m2", JULIA_VERSION = v"1.11.9", nthreads = 1, TOTAL_MEMORY = "24.000 GiB", FREE_MEMORY = "80.922 MiB")

In [24]:
print(results)

(KERNEL = :Darwin, MACHINE = "arm64-apple-darwin24.0.0", CPU_NAME = "apple-m2", JULIA_VERSION = v"1.11.9", nthreads = 1, TOTAL_MEMORY = "24.000 GiB", FREE_MEMORY = "80.922 MiB")

In [25]:
function comp_pi(x, y)
    inside = @. ifelse(x^2 + y^2 <= 1f0, 1f0, 0f0)
    return 4f0 * sum(inside) / Float32(length(x))
end

comp_pi (generic function with 1 method)

In [ ]:
N = 10^7
x_host = rand(Float32, N)
y_host = rand(Float32, N)
x_ra = Reactant.ConcreteRArray(x_host)
y_ra = Reactant.ConcreteRArray(y_host)

In [ ]:
f = Reactant.compile(comp_pi, (x_ra, y_ra))

Reactant compiled function comp_pi (with tag ##comp_pi_reactant#248)

In [ ]:
f(x_ra, y_ra)

Reactant.ConcretePJRTNumber{Float32, 1}(3.1414313f0)

In [ ]:
@benchmark $f($x_ra, $y_ra)

BenchmarkTools.Trial: 345 samples with 1 evaluation per sample.
 Range (min … max):   9.200 μs … 24.499 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     15.594 ms              ┊ GC (median):    0.00%
 Time  (mean ± σ):   14.378 ms ±  4.885 ms  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▃                                      ▅▅█▆▁                 
  █▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▃▂▁▃▃▇▆██████▆▄▄▃▃▃▂▂▁▁▁▁▁▃ ▃
  9.2 μs          Histogram: frequency by time        22.3 ms <

 Memory estimate: 400 bytes, allocs estimate: 14.